# OGC API Records — Metadata Records via DynaStore

Demonstrates OGC API — Records (Parts 1–2) against DynaStore.

## What this notebook covers

1. Catalog setup via the `public_open_data` preset.
2. List catalogs and collections via the Records service.
3. Ingest records (geometry-free `Feature` items).
4. Read a single record and a paged collection.
5. Teardown.

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`),
`DYNASTORE_SYSADMIN_TOKEN` or `DYNASTORE_TOKEN`.

**Routes verified against** `records_service.py` `_register_routes()`:
- `GET /records/` → landing page
- `GET /records/conformance` → conformance
- `GET /records/catalogs` → catalog list
- `GET /records/catalogs/{cat}/collections` → 200
- `GET /records/catalogs/{cat}/collections/{col}` → 200
- `POST /records/catalogs/{cat}/collections/{col}/items` → 201
- `GET /records/catalogs/{cat}/collections/{col}/items` → 200
- `GET /records/catalogs/{cat}/collections/{col}/items/{id}` → 200

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

if not TOKEN:
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                TOKEN = _r.json().get("access_token", "")
        except Exception:
            pass

HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
client = httpx.Client(base_url=BASE, headers=HEADERS, timeout=60.0)

CATALOG_ID = "nb03-records-demo"
COLL_ID = "thesaurus-terms"

def check(r, label="", expected=(200, 201)):
    msg = f"{label}: {r.status_code}"
    if r.status_code not in expected:
        msg += f" — {r.text[:300]}"
    print(msg)
    assert r.status_code in expected, msg
    return r

print(f"Base URL   : {BASE}")
print(f"Catalog    : {CATALOG_ID}")
print(f"Collection : {COLL_ID}")

## 1. Catalog setup via `public_open_data` preset

In [ ]:
r = client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "OGC Records Demo",
    "description": "Ephemeral catalog for the OGC API Records notebook.",
})
check(r, "Create catalog", expected=(200, 201, 409))

r = client.post(f"/configs/catalogs/{CATALOG_ID}/presets/public_open_data")
check(r, "Apply preset public_open_data", expected=(200, 201, 409))
print("Catalog ready.")

## 2. Records service landing page and conformance

In [ ]:
r = client.get("/records/")
check(r, "Landing page")
lp = r.json()
print(f"Title: {lp.get('title')}")
print(f"Links: {len(lp.get('links', []))}")

In [ ]:
r = client.get("/records/conformance")
check(r, "Conformance")
conf = r.json()
uris = conf.get("conformsTo", [])
print(f"Conformance URIs ({len(uris)}):")
for u in uris[:5]:
    print(f"  {u}")

In [ ]:
# List catalogs visible to the Records service
r = client.get("/records/catalogs")
check(r, "List catalogs")
catalogs = r.json()
catalog_ids = [c["id"] for c in catalogs.get("catalogs", catalogs if isinstance(catalogs, list) else [])]
print(f"Visible catalogs: {catalog_ids[:5]}")

## 3. Create RECORDS-type collection

The collection is created via the STAC endpoint (catalog-creation API),
then the Records service serves it under `/records/catalogs/{cat}/collections/{col}`.

In [ ]:
r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_ID,
    "type": "Collection",
    "stac_version": "1.1.0",
    "title": "Thesaurus Terms",
    "description": "Controlled-vocabulary metadata records — no spatial component.",
    "keywords": ["thesaurus", "records", "metadata"],
    "license": "CC-BY-4.0",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [[None, None]]}
    },
    "links": [],
})
check(r, "Create collection", expected=(200, 201))
print(f"Collection id: {r.json().get('id')}")

In [ ]:
# List collections via Records service
r = client.get(f"/records/catalogs/{CATALOG_ID}/collections")
check(r, "List collections via Records")
colls = r.json()
coll_ids = [c["id"] for c in colls.get("collections", [])]
print(f"Collections: {coll_ids}")

In [ ]:
# Read collection via Records service
r = client.get(f"/records/catalogs/{CATALOG_ID}/collections/{COLL_ID}")
check(r, "Read collection via Records")
coll = r.json()
print(f"Collection id={coll.get('id')}  title={coll.get('title')}")

## 4. Ingest and read records

Records are `Feature(geometry=None, properties={…})` at the storage layer.
The Records service returns OGC API — Records resource shapes.

In [ ]:
records = [
    {
        "type": "Feature",
        "id": f"term-{i:03d}",
        "geometry": None,
        "properties": {
            "title": f"Term {i}",
            "description": "Controlled-vocabulary entry.",
            "broader": f"root-{i % 3}",
            "language": "en",
        },
    }
    for i in range(1, 6)
]
payload = {"type": "FeatureCollection", "features": records}

r = client.post(
    f"/records/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items",
    content=json.dumps(payload),
    headers={"Content-Type": "application/json"},
)
check(r, "Ingest records", expected=(200, 201))
print(f"Inserted {len(records)} records.")

In [ ]:
# List records (paged)
r = client.get(
    f"/records/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items",
    params={"limit": 3},
)
check(r, "List records (limit=3)")
out = r.json()
print(f"Returned: {len(out.get('features', []))} records")
for feat in out.get("features", []):
    print(f"  id={feat.get('id')}  title={feat.get('properties', {}).get('title')}")

In [ ]:
# Read single record
RECORD_ID = "term-001"
r = client.get(
    f"/records/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items/{RECORD_ID}"
)
check(r, f"Read record {RECORD_ID}")
rec = r.json()
assert rec.get("id") == RECORD_ID, f"Expected id={RECORD_ID}, got {rec.get('id')}"
print(f"Record: id={rec.get('id')}  title={rec.get('properties', {}).get('title')}")

## Teardown

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ID}")
print(f"Delete collection: {r.status_code}")

r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
print(f"Delete catalog: {r.status_code}")

client.close()
print("Teardown complete.")